# Child Malnutrition Risk Predictor — Machine Learning

**Author:** Muhammad Masoom Khan  
**Model:** Random Forest Classifier  
**Data:** Synthetic dataset modelled on MICS Pakistan 2019-20 survey structure

---

The idea behind this came from a practical problem: MUAC screening requires a trained person to physically measure each child. In low-coverage areas, this creates a bottleneck. The question I wanted to answer was — can we use household-level information (which is much easier to collect) to flag children who are likely at risk, so that scarce screening resources get directed to the right places first?

This is a classification problem. The target variable is whether a child has moderate or severe acute malnutrition.

## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve)
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
PURPLE='#6c5ce7'; BLUE='#6495ed'; GREEN='#00b894'; RED='#e17055'; ORANGE='#fdcb6e'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_csv('data/malnutrition_ml_dataset.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
vc = df['malnutrition_risk'].value_counts()
for v, c in vc.items():
    print(f"  {'High risk' if v==1 else 'Low risk'}: {c:,} ({c/len(df)*100:.1f}%)")
df.head()

## 2. Feature Engineering

The categorical variables need encoding. I also create a couple of interaction terms — wealth + food insecurity combined, and a composite access score.

In [ ]:
# Encode categoricals
wealth_map = {'Poorest':4,'Poor':3,'Middle':2,'Rich':1,'Richest':0}
water_map  = {'Piped':0,'Borehole':1,'Surface':2,'Rainwater':3}

df['wealth_enc'] = df['wealth_index'].map(wealth_map)
df['water_enc']  = df['water_source'].map(water_map)

# Interaction: combined deprivation score
df['deprivation_score'] = (df['wealth_enc']/4 * 0.5 +
                           df['food_insecurity_score']/27 * 0.5)

# Access score
df['access_score'] = (df['distance_health_km'] / 60 * 0.6 +
                      df['female_headed'] * 0.4)

FEATURES = ['age_months','sex','hh_size','wealth_enc','food_insecurity_score',
            'water_enc','mother_education','mother_muac','distance_health_km',
            'female_headed','birth_order','antenatal_visits',
            'deprivation_score','access_score']

X = df[FEATURES]
y = df['malnutrition_risk']

print("Feature set ready.")
print(f"Features: {len(FEATURES)}")
print(f"\nCorrelation with target (top 8):")
corrs = X.corrwith(y).abs().sort_values(ascending=False)
print(corrs.head(8).round(3).to_string())

## 3. Train/Test Split & Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Training set : {len(X_train):,} records")
print(f"Test set     : {len(X_test):,} records")
print(f"Positive rate (train): {y_train.mean()*100:.1f}%")
print(f"Positive rate (test) : {y_test.mean()*100:.1f}%")

# Train — I settled on these hyperparameters after a grid search
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',   # important: handles class imbalance
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print("\nModel trained.")

# Cross-validation to check for overfitting
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='f1')
print(f"5-fold CV F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

## 4. Evaluation — How Good Is It?

In [ ]:
y_pred      = rf.predict(X_test)
y_pred_proba= rf.predict_proba(X_test)[:,1]

print("Classification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Low Risk','High Risk']))

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor='#1a1d23')
for ax in axes: ax.set_facecolor('#1a1d23')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=['Low Risk','High Risk'],
            yticklabels=['Low Risk','High Risk'],
            annot_kws={'size':14, 'weight':'bold'})
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold', pad=12)
axes[0].set_ylabel('Actual', fontsize=11)
axes[0].set_xlabel('Predicted', fontsize=11)

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color=PURPLE, lw=2.5, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1], '--', color='gray', lw=1, alpha=0.6)
axes[1].fill_between(fpr, tpr, alpha=0.08, color=PURPLE)
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold', pad=12)
axes[1].legend(facecolor='#2f3436', labelcolor='white', fontsize=11)

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test, y_pred_proba)
axes[2].plot(rec, prec, color=GREEN, lw=2.5)
axes[2].fill_between(rec, prec, alpha=0.08, color=GREEN)
axes[2].axhline(y_test.mean(), color='gray', ls='--', lw=1, alpha=0.6, label='Baseline')
axes[2].set_xlabel('Recall', fontsize=11)
axes[2].set_ylabel('Precision', fontsize=11)
axes[2].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold', pad=12)
axes[2].legend(facecolor='#2f3436', labelcolor='white')

plt.tight_layout()
plt.savefig('outputs/01_model_performance.png', dpi=150, bbox_inches='tight', facecolor='#1a1d23')
plt.show()

## 5. Feature Importance — What Actually Drives Risk?

In [ ]:
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#1a1d23')
for ax in axes: ax.set_facecolor('#1a1d23')

# Feature importance bar chart
bar_colors = [PURPLE if v == fi.max() else BLUE for v in fi.values]
axes[0].barh(fi.index, fi.values, color=bar_colors, edgecolor='none', height=0.7)
axes[0].set_xlabel('Importance Score', fontsize=11)
axes[0].set_title('Feature Importance\n(Random Forest — Gini impurity)', fontsize=12, fontweight='bold')
for i, v in enumerate(fi.values):
    axes[0].text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=8)

# Risk score distribution by actual outcome
df_test = X_test.copy()
df_test['actual'] = y_test.values
df_test['risk_score'] = y_pred_proba

axes[1].hist(df_test[df_test['actual']==0]['risk_score'], bins=30,
             alpha=0.6, color=BLUE, label='Actual: Low Risk', edgecolor='none')
axes[1].hist(df_test[df_test['actual']==1]['risk_score'], bins=30,
             alpha=0.6, color=RED, label='Actual: High Risk', edgecolor='none')
axes[1].axvline(0.5, color='white', ls='--', lw=1.5, alpha=0.7, label='Decision threshold')
axes[1].set_xlabel('Predicted Risk Score', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('Predicted Risk Score Distribution\nby Actual Outcome', fontsize=12, fontweight='bold')
axes[1].legend(facecolor='#2f3436', labelcolor='white')

plt.tight_layout()
plt.savefig('outputs/02_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#1a1d23')
plt.show()

## 6. Summary & Practical Use

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

print("=" * 55)
print("  MODEL SUMMARY — CHILD MALNUTRITION RISK PREDICTOR")
print("=" * 55)
print(f"  Model type    : Random Forest Classifier")
print(f"  Features used : {len(FEATURES)}")
print(f"  Training size : {len(X_train):,} records")
print(f"  Test size     : {len(X_test):,} records")
print()
print("  Performance on held-out test set:")
print(f"  Accuracy      : {(y_pred == y_test).mean()*100:.1f}%")
print(f"  Precision     : {precision_score(y_test,y_pred):.3f}")
print(f"  Recall        : {recall_score(y_test,y_pred):.3f}")
print(f"  F1 Score      : {f1_score(y_test,y_pred):.3f}")
print(f"  AUC-ROC       : {roc_auc:.3f}")
print()
print("  Top 3 predictive features:")
for feat, imp in fi.sort_values(ascending=False).head(3).items():
    print(f"    {feat:<30} {imp:.4f}")
print()
print("  Practical application:")
print("  Households with deprivation_score > 0.6 and distance_health_km > 15")
print(f"  have {df[(df['deprivation_score']>0.6)&(df['distance_health_km']>15)]['malnutrition_risk'].mean()*100:.0f}% malnutrition rate vs")
print(f"  {df['malnutrition_risk'].mean()*100:.0f}% overall — useful for targeting screening visits.")